# Descarga de Datos Halts Oficiales

Este notebook verifica que la descarga y consolidacion de `halts` oficiales en `D:\Halts` esta bien formada y que el `master` multisource cruza correctamente contra el universo actual de SmallCaps.

Objetivos de verificacion:

- comprobar que existen los artefactos raw y processed esperados
- comprobar el schema de `Nasdaq`, `NYSE`, `SEC` y `multisource`
- medir cobertura del universo contra `halts`
- validar que el cruce por ticker no esta roto
- dejar tablas listas para uso en `Agent03` y `046`


In [1]:
from pathlib import Path
import pandas as pd
import json
from IPython.display import display

HALTS_ROOT = Path(r"D:\Halts")
RAW_ROOT = HALTS_ROOT / "raw"
PROC_ROOT = HALTS_ROOT / "processed"
UNIVERSE_CSV = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_2005_2026_upper_excluding_ohlcv_1m_missing_vs_daily_ge_2B_or_null.csv")
UNIVERSE_PARQUET = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_2005_2026_upper_excluding_ohlcv_1m_missing_vs_daily_ge_2B_or_null.parquet")

NASDAQ_PARQUET = PROC_ROOT / "halts_master_nasdaq_for_run_dates.parquet"
NYSE_PARQUET = PROC_ROOT / "halts_master_nyse_1y.parquet"
SEC_PARQUET = PROC_ROOT / "halts_master_sec.parquet"
MULTI_PARQUET = PROC_ROOT / "halts_master_multisource.parquet"
MULTI_COVERAGE_CSV = PROC_ROOT / "universe_vs_halts_coverage_multisource.csv"
SUMMARY_CSV = PROC_ROOT / "halts_master_multisource_summary.csv"

print("HALTS_ROOT:", HALTS_ROOT)
print("UNIVERSE_CSV exists:", UNIVERSE_CSV.exists())
print("UNIVERSE_PARQUET exists:", UNIVERSE_PARQUET.exists())
print("NASDAQ_PARQUET exists:", NASDAQ_PARQUET.exists())
print("NYSE_PARQUET exists:", NYSE_PARQUET.exists())
print("SEC_PARQUET exists:", SEC_PARQUET.exists())
print("MULTI_PARQUET exists:", MULTI_PARQUET.exists())
print("MULTI_COVERAGE_CSV exists:", MULTI_COVERAGE_CSV.exists())
print("SUMMARY_CSV exists:", SUMMARY_CSV.exists())


HALTS_ROOT: D:\Halts
UNIVERSE_CSV exists: True
UNIVERSE_PARQUET exists: True
NASDAQ_PARQUET exists: True
NYSE_PARQUET exists: True
SEC_PARQUET exists: True
MULTI_PARQUET exists: True
MULTI_COVERAGE_CSV exists: True
SUMMARY_CSV exists: True


## Inventario de artefactos

Esta seccion comprueba que la estructura de `D:\Halts` contiene lo esperado:

- raw Nasdaq
- raw NYSE
- raw SEC
- masters processed por fuente
- master multisource
- coverage del universo


In [2]:
def files_table(root: Path, pattern: str = "*"):
    rows = []
    for fp in sorted(root.rglob(pattern)):
        if fp.is_file():
            rows.append({
                "path": str(fp),
                "size_bytes": fp.stat().st_size,
            })
    return pd.DataFrame(rows)

raw_files = files_table(RAW_ROOT)
proc_files = files_table(PROC_ROOT)

print("raw files:", len(raw_files))
print("processed files:", len(proc_files))
display(raw_files.head(30))
display(proc_files)


raw files: 5681
processed files: 20


,path,size_bytes
0,D:\Halts\raw\nasdaq\https_www_nasdaqtrader_com...,49891
1,D:\Halts\raw\nasdaq\TradingHaltHistory.html,52338
2,D:\Halts\raw\nasdaq_rss_by_date\01022004.xml,448
3,D:\Halts\raw\nasdaq_rss_by_date\01022008.xml,10671
4,D:\Halts\raw\nasdaq_rss_by_date\01022009.xml,1907
5,D:\Halts\raw\nasdaq_rss_by_date\01022013.xml,1940
6,D:\Halts\raw\nasdaq_rss_by_date\01022014.xml,36993
7,D:\Halts\raw\nasdaq_rss_by_date\01022015.xml,68554
8,D:\Halts\raw\nasdaq_rss_by_date\01022018.xml,29002
9,D:\Halts\raw\nasdaq_rss_by_date\01022019.xml,33227


,path,size_bytes
0,D:\Halts\processed\halts_master.csv,110
1,D:\Halts\processed\halts_master.parquet,4848
2,D:\Halts\processed\halts_master_multisource.csv,68688857
3,D:\Halts\processed\halts_master_multisource.pa...,9737694
4,D:\Halts\processed\halts_master_multisource_su...,102
5,D:\Halts\processed\halts_master_nasdaq_for_run...,54374003
6,D:\Halts\processed\halts_master_nasdaq_for_run...,6548273
7,D:\Halts\processed\halts_master_nyse_1y.csv,2664776
8,D:\Halts\processed\halts_master_nyse_1y.parquet,352636
9,D:\Halts\processed\halts_master_sec.csv,365706


## Universe baseline

Se fija el universo base y se mide:

- filas del CSV
- tickers unicos
- rango alfabetico de tickers


In [3]:
u = pd.read_csv(UNIVERSE_CSV)
u["ticker"] = u["ticker"].astype(str).str.upper().str.strip()
u_unique = u[["ticker"]].drop_duplicates().reset_index(drop=True)

summary_universe = pd.DataFrame([{
    "rows_csv": int(len(u)),
    "unique_tickers": int(u_unique["ticker"].nunique()),
    "min_ticker": u_unique["ticker"].min(),
    "max_ticker": u_unique["ticker"].max(),
}])

display(summary_universe)
display(u_unique.head(20))


,rows_csv,unique_tickers,min_ticker,max_ticker
0,12133,12132,AAA,ZZ


,ticker
0,AAA
1,AABA
2,AAC
3,AACB
4,AACI
5,AACQ
6,AACT
7,AAGR
8,AAI
9,AAIC


## Schema y calidad basica por fuente

Se inspeccionan schemas y conteos de filas para:

- Nasdaq
- NYSE
- SEC
- multisource


In [4]:
frames = {}
for name, fp in {
    "nasdaq": NASDAQ_PARQUET,
    "nyse": NYSE_PARQUET,
    "sec": SEC_PARQUET,
    "multisource": MULTI_PARQUET,
}.items():
    if fp.exists():
        frames[name] = pd.read_parquet(fp)
        print("\nSOURCE", name)
        print("rows", len(frames[name]))
        print("cols", frames[name].columns.tolist())
        display(frames[name].head(3))



SOURCE nasdaq
rows 119630
cols ['ticker', 'halt_date', 'halt_start_et', 'resume_quote_et', 'resume_trade_et', 'halt_code', 'halt_type', 'title', 'url_source', 'item_link', 'source_priority', 'raw_description_text']


,ticker,halt_date,halt_start_et,resume_quote_et,resume_trade_et,halt_code,halt_type,title,url_source,item_link,source_priority,raw_description_text
0,NaN,NaT,NaT,NaT,NaT,NaN,None,RTLX,https://www.nasdaqtrader.com/rss.aspx?feed=tra...,,2,Issue Symbol Issue Name Mkt Reason Code ...
1,NaN,NaT,NaT,NaT,NaT,NaN,None,FINB,https://www.nasdaqtrader.com/rss.aspx?feed=tra...,,2,Issue Symbol Issue Name Mkt Reason Code ...
2,NaN,NaT,NaT,NaT,NaT,NaN,None,TDN,https://www.nasdaqtrader.com/rss.aspx?feed=tra...,,2,Issue Symbol Issue Name Mkt Reason Code ...



SOURCE nyse
rows 13178
cols ['source', 'source_priority', 'ticker', 'issuer_name', 'listing_exchange', 'halt_date', 'halt_start_et', 'resume_quote_et', 'resume_trade_et', 'halt_code', 'halt_type', 'raw_reason', 'release_no', 'item_link', 'url_source', 'is_sec_suspension']


,source,source_priority,ticker,issuer_name,listing_exchange,halt_date,halt_start_et,resume_quote_et,resume_trade_et,halt_code,halt_type,raw_reason,release_no,item_link,url_source,is_sec_suspension
0,nyse,1,PED,PEDEVCO Corp.,NYSE American,2026-03-12,2026-03-12 19:50:19,NaT,NaT,None,Corporate Action,Corporate Action,None,None,https://www.nyse.com/api/trade-halts/historica...,False
1,nyse,1,KXIN,Kaixin Holdings Ordinary Shares,Nasdaq,2026-03-12,2026-03-12 19:50:00,NaT,NaT,None,News pending,News pending,None,None,https://www.nyse.com/api/trade-halts/historica...,False
2,nyse,1,STFS,Star Fashion Culture Holdings Limited Class A ...,Nasdaq,2026-03-12,2026-03-12 19:50:00,NaT,NaT,None,News pending,News pending,None,None,https://www.nyse.com/api/trade-halts/historica...,False



SOURCE sec
rows 1346
cols ['source', 'source_priority', 'ticker', 'issuer_name', 'listing_exchange', 'halt_date', 'halt_start_et', 'resume_quote_et', 'resume_trade_et', 'halt_code', 'halt_type', 'raw_reason', 'release_no', 'item_link', 'url_source', 'is_sec_suspension']


,source,source_priority,ticker,issuer_name,listing_exchange,halt_date,halt_start_et,resume_quote_et,resume_trade_et,halt_code,halt_type,raw_reason,release_no,item_link,url_source,is_sec_suspension
0,sec,1,TCGL,TechCreate Group Ltd.,None,2026-02-02,NaT,NaT,NaT,SEC,SEC suspension,SEC suspension,34-104763,https://www.sec.gov/files/litigation/suspensio...,https://www.sec.gov/enforcement-litigation/tra...,True
1,sec,1,JMG,JM Group Limited,None,2026-01-15,NaT,NaT,NaT,SEC,SEC suspension,SEC suspension,34-104613,https://www.sec.gov/files/litigation/suspensio...,https://www.sec.gov/enforcement-litigation/tra...,True
2,sec,1,NaN,Magnitude International Ltd,None,2025-12-05,NaT,NaT,NaT,SEC,SEC suspension,SEC suspension,34-104317,https://www.sec.gov/files/litigation/suspensio...,https://www.sec.gov/enforcement-litigation/tra...,True



SOURCE multisource
rows 133116
cols ['source', 'source_priority', 'ticker', 'issuer_name', 'listing_exchange', 'halt_date', 'halt_start_et', 'resume_quote_et', 'resume_trade_et', 'halt_code', 'halt_type', 'raw_reason', 'release_no', 'item_link', 'url_source', 'is_sec_suspension']


,source,source_priority,ticker,issuer_name,listing_exchange,halt_date,halt_start_et,resume_quote_et,resume_trade_et,halt_code,halt_type,raw_reason,release_no,item_link,url_source,is_sec_suspension
0,sec,1,<NA>,"Garcis U.S.A., Inc.",NaN,1995-10-13,NaT,NaT,NaT,SEC,SEC suspension,SEC suspension,34-36366,https://www.sec.gov/files/litigation/admin/343...,https://www.sec.gov/enforcement-litigation/tra...,True
1,sec,1,<NA>,"Environmental Chemicals Group, Inc.",NaN,1995-12-12,NaT,NaT,NaT,SEC,SEC suspension,SEC suspension,34-36571,https://www.sec.gov/enforcement-litigation/tra...,https://www.sec.gov/enforcement-litigation/tra...,True
2,sec,1,<NA>,"The Enstar Group, Inc.",NaN,1996-03-29,NaT,NaT,NaT,SEC,SEC suspension,SEC suspension,34-37043,https://www.sec.gov/files/litigation/admin/343...,https://www.sec.gov/enforcement-litigation/tra...,True


## Resumen por fuente

Se comprueba cuantas filas y cuantos tickers no nulos tiene cada source del consolidado.


In [5]:
if SUMMARY_CSV.exists():
    summary = pd.read_csv(SUMMARY_CSV)
    display(summary)
else:
    multi = frames["multisource"].copy()
    tmp = pd.DataFrame({
        "source": sorted(multi["source"].dropna().astype(str).unique().tolist())
    })
    rows = []
    for src, grp in multi.groupby("source", dropna=False):
        rows.append({
            "source": src,
            "rows": int(len(grp)),
            "tickers_nonnull": int(grp["ticker"].notna().sum()),
        })
    display(pd.DataFrame(rows))


,source,rows,tickers_nonnull
0,nasdaq,119630,119619
1,nyse,13178,13178
2,sec,1346,250
3,all,133116,132009


## Coverage del universo contra multisource

Aqui medimos cuanta parte del universo actual tiene al menos un `halt` en el master multisource.


In [6]:
if MULTI_COVERAGE_CSV.exists():
    coverage = pd.read_csv(MULTI_COVERAGE_CSV)
else:
    multi = frames["multisource"].copy()
    multi["ticker"] = multi["ticker"].astype("string").str.upper().str.strip()
    agg = multi[multi["ticker"].notna()].groupby("ticker", dropna=False).agg(
        halt_events_count=("ticker", "size"),
        source_count=("source", "nunique"),
        sources=("source", lambda s: ",".join(sorted(set(map(str, s.dropna()))))),
        first_halt_date=("halt_date", "min"),
        last_halt_date=("halt_date", "max"),
    ).reset_index()
    coverage = u_unique.merge(agg, on="ticker", how="left")
    coverage["has_halt_data"] = coverage["halt_events_count"].fillna(0).gt(0)

coverage_summary = pd.DataFrame([{
    "universe_tickers": int(len(coverage)),
    "tickers_with_halt_data": int(coverage["has_halt_data"].sum()),
    "tickers_without_halt_data": int((~coverage["has_halt_data"]).sum()),
    "pct_with_halt_data": round(100 * coverage["has_halt_data"].mean(), 2),
    "halt_events_total_in_universe": int(coverage["halt_events_count"].fillna(0).sum()),
}])

display(coverage_summary)


,universe_tickers,tickers_with_halt_data,tickers_without_halt_data,pct_with_halt_data,halt_events_total_in_universe
0,12132,7955,4177,65.57,81773


## Top tickers del universo con mas halts

Esto sirve para verificar que el cruce no esta vacio ni mal alineado.


In [7]:
with_halts = coverage[coverage["has_halt_data"]].copy()
with_halts = with_halts.sort_values(["halt_events_count", "source_count", "ticker"], ascending=[False, False, True])

display(with_halts.head(50))


,ticker,halt_events_count,source_count,sources,first_halt_date,last_halt_date,has_halt_data
9110,RDIB,1206.0,2.0,"nasdaq,nyse",2013-10-01,2026-02-09,True
5992,KELYB,944.0,2.0,"nasdaq,nyse",2013-09-26,2026-02-06,True
582,AMSGP,797.0,1.0,nasdaq,2014-07-10,2016-06-15,True
6544,LTRPB,510.0,1.0,nasdaq,2014-09-16,2023-10-27,True
8507,PHII,504.0,1.0,nasdaq,2013-09-12,2019-03-15,True
7996,OGCP,290.0,2.0,"nasdaq,nyse",2013-10-09,2026-02-12,True
4019,FISK,280.0,2.0,"nasdaq,nyse",2013-10-10,2025-06-13,True
11709,WINS,279.0,1.0,nasdaq,2004-08-30,2020-08-18,True
11090,UAG,241.0,1.0,nasdaq,2013-12-16,2019-11-19,True
9190,RGC,238.0,2.0,"nasdaq,nyse",2017-11-28,2026-03-09,True


## Tickers del universo sin halt data

No implica error necesariamente.

Puede significar:

- no hubo halt en las fuentes oficiales descargadas
- no hubo coincidencia por ticker
- o la cobertura de esa fuente no alcanza ese caso


In [8]:
without_halts = coverage[~coverage["has_halt_data"]].copy()
display(without_halts.head(50))


,ticker,halt_events_count,source_count,sources,first_halt_date,last_halt_date,has_halt_data
0,AAA,NaN,NaN,NaN,NaN,NaN,False
3,AACB,NaN,NaN,NaN,NaN,NaN,False
5,AACQ,NaN,NaN,NaN,NaN,NaN,False
6,AACT,NaN,NaN,NaN,NaN,NaN,False
8,AAI,NaN,NaN,NaN,NaN,NaN,False
9,AAIC,NaN,NaN,NaN,NaN,NaN,False
11,AAM,NaN,NaN,NaN,NaN,NaN,False
14,AAMI,NaN,NaN,NaN,NaN,NaN,False
20,AAQC,NaN,NaN,NaN,NaN,NaN,False
25,AAUC,NaN,NaN,NaN,NaN,NaN,False


## Consistencia por source dentro del coverage

Se revisa cuantos tickers del universo tienen:

- solo Nasdaq
- solo NYSE
- solo SEC
- combinaciones de sources


In [9]:
source_mix = (
    coverage[coverage["has_halt_data"]]
    .groupby("sources", dropna=False)
    .agg(
        tickers=("ticker", "nunique"),
        halt_events_total=("halt_events_count", "sum"),
    )
    .reset_index()
    .sort_values(["tickers", "halt_events_total"], ascending=[False, False])
)

display(source_mix.head(20))


,sources,tickers,halt_events_total
0,nasdaq,6247,42056.0
1,"nasdaq,nyse",1678,39536.0
3,nyse,23,34.0
4,sec,4,4.0
2,"nasdaq,nyse,sec",3,143.0


## Sanity checks de columnas clave

Se revisa que el master multisource conserve las columnas minimas necesarias para Agent03 / 046:

- `ticker`
- `halt_date`
- `halt_start_et`
- `resume_trade_et`
- `halt_type`
- `source`


In [10]:
required_cols = [
    "source", "ticker", "halt_date", "halt_start_et", "resume_trade_et", "halt_type"
]

multi = frames["multisource"].copy()
check = pd.DataFrame([{
    "column": c,
    "exists": c in multi.columns,
    "nonnull_rows": int(multi[c].notna().sum()) if c in multi.columns else 0,
} for c in required_cols])

display(check)


,column,exists,nonnull_rows
0,source,True,133116
1,ticker,True,132009
2,halt_date,True,133105
3,halt_start_et,True,131759
4,resume_trade_et,True,129747
5,halt_type,True,132572


## Muestra operativa para contraste con Agent03

Esta tabla deja una muestra lista para contraste causal rapido.


In [11]:
sample_cols = [
    c for c in [
        "source", "ticker", "issuer_name", "halt_date", "halt_start_et",
        "resume_trade_et", "halt_type", "halt_code", "release_no", "url_source"
    ] if c in multi.columns
]

sample_multi = multi[multi["ticker"].notna()].copy().sort_values(["halt_date", "source", "ticker"], ascending=[False, True, True])
display(sample_multi[sample_cols].head(100))


,source,ticker,issuer_name,halt_date,halt_start_et,resume_trade_et,halt_type,halt_code,release_no,url_source
133104,nasdaq,FTB$C,NaN,2068-04-15,2068-04-15 11:33:42,2011-05-18 09:30:58,NaN,NaN,NaN,https://www.nasdaqtrader.com/rss.aspx?feed=tra...
133099,nasdaq,PIY,NaN,2046-10-01,2046-10-01 10:20:05,2020-03-12 10:25:05,NaN,NaN,NaN,https://www.nasdaqtrader.com/rss.aspx?feed=tra...
133100,nasdaq,PIY,NaN,2046-10-01,2046-10-01 09:30:28,2020-03-30 09:35:28,NaN,NaN,NaN,https://www.nasdaqtrader.com/rss.aspx?feed=tra...
133101,nasdaq,PIY,NaN,2046-10-01,2046-10-01 11:08:24,2020-04-01 11:13:35,NaN,NaN,NaN,https://www.nasdaqtrader.com/rss.aspx?feed=tra...
133102,nasdaq,PIY,NaN,2046-10-01,2046-10-01 08:43:43,2020-05-14 00:00:01,NaN,NaN,NaN,https://www.nasdaqtrader.com/rss.aspx?feed=tra...
...,...,...,...,...,...,...,...,...,...,...
133018,nyse,ELPW,Elong Power Holding Limited Class A Ordinary S...,2026-03-11,2026-03-11 19:50:00,NaT,News pending,NaN,NaN,https://www.nyse.com/api/trade-halts/historica...
133019,nyse,ELPW,Elong Power Holding Limited Class A Ordinary S...,2026-03-11,2026-03-11 19:50:00,2026-03-12 09:00:00,News pending,NaN,NaN,https://www.nyse.com/api/trade-halts/historica...
133020,nyse,GSUN,Golden Sun Technology Group Limited Class A Or...,2026-03-11,2026-03-11 10:15:38,2026-03-11 10:20:38,LULD pause,NaN,NaN,https://www.nyse.com/api/trade-halts/historica...
133021,nyse,IBG,Innovation Beverage Group Limited Ordinary Shares,2026-03-11,2026-03-11 10:23:36,2026-03-11 10:28:36,LULD pause,NaN,NaN,https://www.nyse.com/api/trade-halts/historica...


## Conclusion tecnica

Si este notebook sale bien, se considera verificado que:

- `D:\Halts` contiene raw y processed esperados
- el `master multisource` existe y es legible
- el universo cruza correctamente contra el master
- la cobertura de `halts` esta medida y trazable
- el bloque de `halts` ya esta listo para integrarse en `046_agent3_halt_contrast.py`


In [12]:
from pathlib import Path
import pandas as pd
from IPython.display import display

HALTS_ROOT = Path(r"D:\Halts")
PROC_ROOT = HALTS_ROOT / "processed"
UNIVERSE_CSV = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\tickers_2005_2026_upper_excluding_ohlcv_1m_missing_vs_daily_ge_2B_or_null.csv")

NASDAQ_PARQUET = PROC_ROOT / "halts_master_nasdaq_for_run_dates.parquet"
NYSE_PARQUET = PROC_ROOT / "halts_master_nyse_1y.parquet"
SEC_PARQUET = PROC_ROOT / "halts_master_sec.parquet"
MULTI_PARQUET = PROC_ROOT / "halts_master_multisource.parquet"
MULTI_COVERAGE_CSV = PROC_ROOT / "universe_vs_halts_coverage_multisource.csv"
SUMMARY_CSV = PROC_ROOT / "halts_master_multisource_summary.csv"

u = pd.read_csv(UNIVERSE_CSV)
u["ticker"] = u["ticker"].astype(str).str.upper().str.strip()
u_unique = u[["ticker"]].drop_duplicates().reset_index(drop=True)

frames = {
    "nasdaq": pd.read_parquet(NASDAQ_PARQUET),
    "nyse": pd.read_parquet(NYSE_PARQUET),
    "sec": pd.read_parquet(SEC_PARQUET),
    "multisource": pd.read_parquet(MULTI_PARQUET),
}
coverage = pd.read_csv(MULTI_COVERAGE_CSV)
summary = pd.read_csv(SUMMARY_CSV)

print("reload ok")
print("universe unique tickers:", len(u_unique))
print("multisource rows:", len(frames["multisource"]))
display(summary)

reload ok
universe unique tickers: 12133
multisource rows: 133116


,source,rows,tickers_nonnull
0,nasdaq,119630,119619
1,nyse,13178,13178
2,sec,1346,250
3,all,133116,132009


In [13]:
from IPython.display import display
import pandas as pd

# 1. fuente por fuente
for name, df in frames.items():
    print("\nSOURCE", name)
    print("rows", len(df))
    print("nonnull ticker", int(df["ticker"].notna().sum()) if "ticker" in df.columns else 0)
    display(df.head(3))

# 2. coverage summary
coverage["has_halt_data"] = coverage["halt_events_count"].fillna(0).gt(0)

coverage_summary = pd.DataFrame([{
    "universe_tickers": int(len(coverage)),
    "tickers_with_halt_data": int(coverage["has_halt_data"].sum()),
    "tickers_without_halt_data": int((~coverage["has_halt_data"]).sum()),
    "pct_with_halt_data": round(100 * coverage["has_halt_data"].mean(), 2),
    "halt_events_total_in_universe": int(coverage["halt_events_count"].fillna(0).sum()),
}])
display(coverage_summary)

# 3. top with halts
with_halts = coverage[coverage["has_halt_data"]].copy()
with_halts = with_halts.sort_values(["halt_events_count", "source_count", "ticker"], ascending=[False, False, True])
display(with_halts.head(50))

# 4. without halts
without_halts = coverage[~coverage["has_halt_data"]].copy()
display(without_halts.head(50))

# 5. source mix
source_mix = (
    coverage[coverage["has_halt_data"]]
    .groupby("sources", dropna=False)
    .agg(
        tickers=("ticker", "nunique"),
        halt_events_total=("halt_events_count", "sum"),
    )
    .reset_index()
    .sort_values(["tickers", "halt_events_total"], ascending=[False, False])
)
display(source_mix)

# 6. sanity checks multisource
multi = frames["multisource"].copy()
required_cols = ["source", "ticker", "halt_date", "halt_start_et", "resume_trade_et", "halt_type"]
check = pd.DataFrame([{
    "column": c,
    "exists": c in multi.columns,
    "nonnull_rows": int(multi[c].notna().sum()) if c in multi.columns else 0,
} for c in required_cols])
display(check)

# 7. sample
sample_cols = [c for c in [
    "source", "ticker", "issuer_name", "halt_date", "halt_start_et",
    "resume_trade_et", "halt_type", "halt_code", "release_no", "url_source"
] if c in multi.columns]
sample_multi = multi[multi["ticker"].notna()].copy().sort_values(["halt_date", "source", "ticker"], ascending=[False,
True, True])
display(sample_multi[sample_cols].head(100))


SOURCE nasdaq
rows 119630
nonnull ticker 0


,ticker,halt_date,halt_start_et,resume_quote_et,resume_trade_et,halt_code,halt_type,title,url_source,item_link,source_priority,raw_description_text
0,NaN,NaT,NaT,NaT,NaT,NaN,None,RTLX,https://www.nasdaqtrader.com/rss.aspx?feed=tra...,,2,Issue Symbol Issue Name Mkt Reason Code ...
1,NaN,NaT,NaT,NaT,NaT,NaN,None,FINB,https://www.nasdaqtrader.com/rss.aspx?feed=tra...,,2,Issue Symbol Issue Name Mkt Reason Code ...
2,NaN,NaT,NaT,NaT,NaT,NaN,None,TDN,https://www.nasdaqtrader.com/rss.aspx?feed=tra...,,2,Issue Symbol Issue Name Mkt Reason Code ...



SOURCE nyse
rows 13178
nonnull ticker 13178


,source,source_priority,ticker,issuer_name,listing_exchange,halt_date,halt_start_et,resume_quote_et,resume_trade_et,halt_code,halt_type,raw_reason,release_no,item_link,url_source,is_sec_suspension
0,nyse,1,PED,PEDEVCO Corp.,NYSE American,2026-03-12,2026-03-12 19:50:19,NaT,NaT,None,Corporate Action,Corporate Action,None,None,https://www.nyse.com/api/trade-halts/historica...,False
1,nyse,1,KXIN,Kaixin Holdings Ordinary Shares,Nasdaq,2026-03-12,2026-03-12 19:50:00,NaT,NaT,None,News pending,News pending,None,None,https://www.nyse.com/api/trade-halts/historica...,False
2,nyse,1,STFS,Star Fashion Culture Holdings Limited Class A ...,Nasdaq,2026-03-12,2026-03-12 19:50:00,NaT,NaT,None,News pending,News pending,None,None,https://www.nyse.com/api/trade-halts/historica...,False



SOURCE sec
rows 1346
nonnull ticker 250


...

,ticker,halt_events_count,source_count,sources,first_halt_date,last_halt_date,has_halt_data
9110,RDIB,1206.0,2.0,"nasdaq,nyse",2013-10-01,2026-02-09,True
5992,KELYB,944.0,2.0,"nasdaq,nyse",2013-09-26,2026-02-06,True
582,AMSGP,797.0,1.0,nasdaq,2014-07-10,2016-06-15,True
6544,LTRPB,510.0,1.0,nasdaq,2014-09-16,2023-10-27,True
8507,PHII,504.0,1.0,nasdaq,2013-09-12,2019-03-15,True
7996,OGCP,290.0,2.0,"nasdaq,nyse",2013-10-09,2026-02-12,True
4019,FISK,280.0,2.0,"nasdaq,nyse",2013-10-10,2025-06-13,True
11709,WINS,279.0,1.0,nasdaq,2004-08-30,2020-08-18,True
11090,UAG,241.0,1.0,nasdaq,2013-12-16,2019-11-19,True
9190,RGC,238.0,2.0,"nasdaq,nyse",2017-11-28,2026-03-09,True


,ticker,halt_events_count,source_count,sources,first_halt_date,last_halt_date,has_halt_data
0,AAA,NaN,NaN,NaN,NaN,NaN,False
3,AACB,NaN,NaN,NaN,NaN,NaN,False
5,AACQ,NaN,NaN,NaN,NaN,NaN,False
6,AACT,NaN,NaN,NaN,NaN,NaN,False
8,AAI,NaN,NaN,NaN,NaN,NaN,False
9,AAIC,NaN,NaN,NaN,NaN,NaN,False
11,AAM,NaN,NaN,NaN,NaN,NaN,False
14,AAMI,NaN,NaN,NaN,NaN,NaN,False
20,AAQC,NaN,NaN,NaN,NaN,NaN,False
25,AAUC,NaN,NaN,NaN,NaN,NaN,False


,sources,tickers,halt_events_total
0,nasdaq,6247,42056.0
1,"nasdaq,nyse",1678,39536.0
3,nyse,23,34.0
4,sec,4,4.0
2,"nasdaq,nyse,sec",3,143.0


,column,exists,nonnull_rows
0,source,True,133116
1,ticker,True,132009
2,halt_date,True,133105
3,halt_start_et,True,131759
4,resume_trade_et,True,129747
5,halt_type,True,132572


,source,ticker,issuer_name,halt_date,halt_start_et,resume_trade_et,halt_type,halt_code,release_no,url_source
133104,nasdaq,FTB$C,NaN,2068-04-15,2068-04-15 11:33:42,2011-05-18 09:30:58,NaN,NaN,NaN,https://www.nasdaqtrader.com/rss.aspx?feed=tra...
133099,nasdaq,PIY,NaN,2046-10-01,2046-10-01 10:20:05,2020-03-12 10:25:05,NaN,NaN,NaN,https://www.nasdaqtrader.com/rss.aspx?feed=tra...
133100,nasdaq,PIY,NaN,2046-10-01,2046-10-01 09:30:28,2020-03-30 09:35:28,NaN,NaN,NaN,https://www.nasdaqtrader.com/rss.aspx?feed=tra...
133101,nasdaq,PIY,NaN,2046-10-01,2046-10-01 11:08:24,2020-04-01 11:13:35,NaN,NaN,NaN,https://www.nasdaqtrader.com/rss.aspx?feed=tra...
133102,nasdaq,PIY,NaN,2046-10-01,2046-10-01 08:43:43,2020-05-14 00:00:01,NaN,NaN,NaN,https://www.nasdaqtrader.com/rss.aspx?feed=tra...
...,...,...,...,...,...,...,...,...,...,...
133018,nyse,ELPW,Elong Power Holding Limited Class A Ordinary S...,2026-03-11,2026-03-11 19:50:00,NaT,News pending,NaN,NaN,https://www.nyse.com/api/trade-halts/historica...
133019,nyse,ELPW,Elong Power Holding Limited Class A Ordinary S...,2026-03-11,2026-03-11 19:50:00,2026-03-12 09:00:00,News pending,NaN,NaN,https://www.nyse.com/api/trade-halts/historica...
133020,nyse,GSUN,Golden Sun Technology Group Limited Class A Or...,2026-03-11,2026-03-11 10:15:38,2026-03-11 10:20:38,LULD pause,NaN,NaN,https://www.nyse.com/api/trade-halts/historica...
133021,nyse,IBG,Innovation Beverage Group Limited Ordinary Shares,2026-03-11,2026-03-11 10:23:36,2026-03-11 10:28:36,LULD pause,NaN,NaN,https://www.nyse.com/api/trade-halts/historica...


  - el parser Nasdaq ya sirve para coverage y análisis general
  - pero todavía no está 100% limpio para usar cada timestamp individual como verdad absoluta
  - especialmente en extremos de muestra/sorting

  Conclusión técnica
  Ahora mismo el notebook dice esto:

  1. Coverage multisource

  - ya está bien
  - utilizable

  2. Top tickers y mix de fuentes

  - ya está bien
  - utilizable

  3. Master multisource

  - ya está muy mejorado
  - pero aún tiene outliers temporales raros en algunos registros Nasdaq

  Qué confiaría ya

  - has_halt_data
  - halt_events_count
  - source_count
  - sources
  - cobertura por ticker
  - mezcla Nasdaq/NYSE/SEC
  - first/last halt en general

  Qué todavía trataría con cautela

  - timestamps exactos de algunos registros Nasdaq extremos
  - muestras ordenadas por halt_date en celdas tipo 21
  - análisis intradía fino si depende de esos outliers raros no filtrados

  Mi veredicto final

  - sí, ahora el notebook ya está mayormente bien
  - sí, el problema principal quedó resuelto
  - queda una limpieza residual del parser Nasdaq para casos raros

  Lo resumiría así:

  - coverage: listo
  - ranking y mix por ticker: listo
  - timestamps Nasdaq edge cases: pendiente de saneo fino

  Si quieres, el siguiente paso es uno de estos dos:

  1. dejarlo así y usarlo ya para análisis de cobertura
  2. o hacer un último saneo del parser Nasdaq para eliminar fechas imposibles (> hoy + 1 día, combinaciones absurdas
     halt_date/resume_trade_et)